# Day 7 Tutorial：ESOL 基线静态审计

## Goal

定位真实基线的 10 个代码单元，核对预存 Notebook、CSV 与 JSON 的内部一致性；本教程不替代真实基线的 Restart + Run All。

## Setup

只读取仓库已有产物，不训练模型、不修改结果、不产生 test 预测。

In [1]:
import csv
import json
from pathlib import Path

def find_repo_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('请从 ML-Learning 仓库内部运行')

repo_root = find_repo_root()
experiment = repo_root / 'curriculum' / 'core' / 'day07_integrated_baseline' / 'reference_baseline'
notebook_path = experiment / 'esol_baseline.ipynb'
metrics_path = experiment / 'results' / 'baseline_metrics.csv'
config_path = experiment / 'results' / 'run_config.json'

print('repository:', repo_root.name)
print('required files exist:', all(path.exists() for path in [notebook_path, metrics_path, config_path]))

repository: ML-practice
required files exist: True


## Steps

先读取真实 Notebook 的代码单元状态，再读取结果契约。

In [2]:
notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
code_cells = [cell for cell in notebook['cells'] if cell['cell_type'] == 'code']
execution_counts = [cell.get('execution_count') for cell in code_cells]
error_outputs = [
    output
    for cell in code_cells
    for output in cell.get('outputs', [])
    if output.get('output_type') == 'error'
]

cell_map = {
    1: '环境、路径、种子与版本',
    2: '加载 ESOL 与 ECFP',
    3: 'shape、有限值与 ID 重叠检查',
    4: '形成训练/验证 X 与 y',
    5: '定义回归指标',
    6: '创建固定候选模型',
    7: 'fit、predict 与 metric',
    8: 'validation 排序和汇总',
    9: '保存 CSV 与 JSON',
    10: '自动断言与证据边界',
}

print('code cells:', len(code_cells))
print('execution counts:', execution_counts)
print('saved errors:', len(error_outputs))
for number, purpose in cell_map.items():
    print(f'{number:>2}: {purpose}')

code cells: 10
execution counts: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
saved errors: 0
 1: 环境、路径、种子与版本
 2: 加载 ESOL 与 ECFP
 3: shape、有限值与 ID 重叠检查
 4: 形成训练/验证 X 与 y
 5: 定义回归指标
 6: 创建固定候选模型
 7: fit、predict 与 metric
 8: validation 排序和汇总
 9: 保存 CSV 与 JSON
10: 自动断言与证据边界


In [3]:
with metrics_path.open(encoding='utf-8', newline='') as handle:
    metric_rows = list(csv.DictReader(handle))
config = json.loads(config_path.read_text(encoding='utf-8'))

valid_rows = [row for row in metric_rows if row['split'] == 'valid']
best_from_csv = min(valid_rows, key=lambda row: float(row['rmse_logS']))['model']

print('metric rows:', len(metric_rows))
print('splits:', sorted({row['split'] for row in metric_rows}))
print('best candidate matches config:', best_from_csv == config['best_model_by_validation_rmse'])

metric rows: 10
splits: ['train', 'valid']
best candidate matches config: True


## Checks

静态契约必须一致，但只能证明文件状态，不能证明学习者刚完成重跑。

In [4]:
assert len(code_cells) == 10
assert execution_counts == list(range(1, 11))
assert error_outputs == []
assert len(metric_rows) == 10
assert {row['split'] for row in metric_rows} == {'train', 'valid'}
assert best_from_csv == config['best_model_by_validation_rmse']
assert 'test' not in {row['split'] for row in metric_rows}
print('static baseline audit passed')

static baseline audit passed


## Next Steps

在正确环境中对 `curriculum/core/day07_integrated_baseline/reference_baseline/esol_baseline.ipynb` 执行 Restart + Run All，记录本次执行状态，再独立完成 `03_exercises.md`。